# Qwen2.5-VL-7B — Türkçe El Yazılı Sınav OCR Spike

Bu notebook, lokal makinedeki 4 GB VRAM kısıtının üstesinden gelmek için Colab T4 GPU'sunda Qwen2.5-VL-7B'yi (4-bit quantize) test eder.

**Amaç:** Türkçe el yazısı sınav kâğıdından soru-cevap JSON yapısı çıkarmak.

**Üç hipotez:**
- H1: Qwen2.5-VL-7B Türkçe el yazısını doğru okur mu?
- H2: Custom JSON prompt'a uyup yapılandırılmış çıkış verir mi?
- H3: Header/footer (öğrenci ismi, okul) gürültüsünü eler mi?

**Önceki spike sonuçları:**
- PaddleOCR-VL-1.5 (lokal, 4 GB VRAM): Basılı metin mükemmel, el yazısı yetersiz
- Qwen2.5-VL-3B (lokal, 4 GB VRAM): VRAM yetmedi, vision encoder OOM

**Çalıştırmadan önce:** Runtime → Change runtime type → **T4 GPU** seçili olmalı.

## 1. GPU kontrolü

In [ ]:
!nvidia-smi

Beklenen: **NVIDIA Tesla T4, 16 GB VRAM**. Eğer farklıysa Runtime > Change runtime type > T4 GPU seç.

## 2. Bağımlılıklar

Colab'da torch + CUDA zaten kurulu. Sadece transformers'ı v5+'a güncelliyoruz, bitsandbytes ve accelerate ekliyoruz.

In [ ]:
# pillow<12 — Colab'ta pre-installed torchvision Pillow 12.x ile uyumsuz (_Ink import hatası).
# Yükleme sonrası Runtime → Restart runtime gerekli.
!pip install -q -U "transformers>=5.0.0" accelerate bitsandbytes "pillow<12"

In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda_available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
    print('vram_gb:', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))

import transformers
print('transformers:', transformers.__version__)

import bitsandbytes as bnb
print('bitsandbytes:', bnb.__version__)

## 3. Sınav görüntüsünü yükle

Aşağıdaki hücreyi çalıştır, **'Choose Files'** ile lokal makineden `ornek3.jpeg` (veya başka sınav kâğıdı fotoğrafı) seç.

In [ ]:
from google.colab import files

uploaded = files.upload()
image_filename = list(uploaded.keys())[0]
print(f'Yüklenen: {image_filename}')

In [ ]:
from PIL import Image

image = Image.open(image_filename).convert('RGB')
print(f'Boyut: {image.size}')
image  # Colab notebook'ta önizleme gösterir

## 4. Qwen2.5-VL-7B model yükleme (4-bit quantize)

İlk çalıştırmada model indirilir (~15 GB → 4-bit ile ~6 GB disk). Bu yaklaşık 5-10 dakika sürer. VRAM kullanımı: ~6 GB.

In [ ]:
import time
from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    Qwen2_5_VLForConditionalGeneration,
)

MODEL_PATH = 'Qwen/Qwen2.5-VL-7B-Instruct'

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)

t0 = time.perf_counter()
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    quantization_config=quant_config,
    device_map='cuda:0',
).eval()
print(f'Model yüklendi: {time.perf_counter() - t0:.1f} sn')

# T4 16 GB'da rahat olmak için image token'ı sınırlı tutalım (ama lokal'de 4 GB'a kıyasla 4x bol).
processor = AutoProcessor.from_pretrained(
    MODEL_PATH,
    min_pixels=256 * 28 * 28,
    max_pixels=1280 * 28 * 28,
)

vram_used = torch.cuda.memory_allocated() / (1024**3)
print(f'VRAM kullanılan: {vram_used:.2f} GB / 16 GB')

## 5. Inference yardımcı fonksiyonu

In [ ]:
def run_inference(image, prompt_text, max_new_tokens=1024):
    messages = [
        {
            'role': 'user',
            'content': [
                {'type': 'image', 'image': image},
                {'type': 'text', 'text': prompt_text},
            ],
        }
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(
        text=[text],
        images=[image],
        padding=True,
        return_tensors='pt',
    ).to(model.device)

    n_input = inputs['input_ids'].shape[-1]
    print(f'[gen] girdi token: {n_input}, max_new: {max_new_tokens}')

    t0 = time.perf_counter()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            repetition_penalty=1.2,
            no_repeat_ngram_size=4,
            do_sample=True,
            temperature=0.1,
            top_p=0.9,
        )
    elapsed = time.perf_counter() - t0
    n_gen = outputs.shape[-1] - n_input
    print(f'[gen] {elapsed:.1f} sn, üretilen {n_gen} token')

    return processor.batch_decode(
        outputs[:, n_input:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

## 6. Spike S1 — Ham OCR (basitten karmaşığa giderken referans noktası)

In [ ]:
S1_PROMPT = (
    'Bu görüntüdeki tüm metni satır satır oku ve yaz. '
    'Hem basılı metni hem de el yazısı kısımları olduğu gibi çıkar. '
    'Türkçe karakterlere dikkat et.'
)

result_s1 = run_inference(image, S1_PROMPT, max_new_tokens=1024)
print('\n' + '=' * 60)
print('S1 — HAM OCR ÇIKIŞI')
print('=' * 60)
print(result_s1)

## 7. Spike S2 — Yapılandırılmış JSON çıkışı (asıl hedef)

In [ ]:
S2_PROMPT = (
    'Bu bir Türkçe açık uçlu sınav kâğıdıdır. Aşağıdaki JSON şemasına uyarak '
    'soru numaralarını ve öğrencinin el yazısı cevaplarını çıkar. '
    'Üst bilgi (öğrenci ismi, okul, sınıf, numara), marj notları ve sayfa numaralarını yok say. '
    'Sadece geçerli JSON döndür, açıklama yazma.\n\n'
    'Şema: [{"question_number": "1", "question_text": "...", "extracted_answer": "..."}]'
)

result_s2 = run_inference(image, S2_PROMPT, max_new_tokens=1024)
print('\n' + '=' * 60)
print('S2 — JSON ÇIKIŞI')
print('=' * 60)
print(result_s2)

In [ ]:
import json

def try_parse_json(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    start = text.find('[')
    end = text.rfind(']')
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end + 1])
        except json.JSONDecodeError:
            return None
    return None

parsed = try_parse_json(result_s2)
if parsed:
    print(f'JSON parse başarılı — {len(parsed)} kayıt:\n')
    print(json.dumps(parsed, ensure_ascii=False, indent=2))
else:
    print('JSON parse BAŞARISIZ — model JSON formatını tutturamadı.')

## 8. Spike S3 — Sadece el yazısı cevaplar (NLP'ye besleyeceğimiz veri)

In [ ]:
S3_PROMPT = (
    'Bu görüntüde basılı (matbu) sorulara öğrenci tarafından elle yazılmış cevaplar var. '
    'Sadece öğrencinin el yazısıyla yazdığı cevapları soru numarasıyla eşleştirip listele. '
    'Format: "1) cevap metni" şeklinde her satır ayrı. '
    'Öğrenci ismi, okul, marj notları gibi şeyleri yazma.'
)

result_s3 = run_inference(image, S3_PROMPT, max_new_tokens=768)
print('\n' + '=' * 60)
print('S3 — SADECE EL YAZISI CEVAPLAR')
print('=' * 60)
print(result_s3)

## 9. Değerlendirme

Üç çıkışı karşılaştır:

1. **S1 ham OCR:** Türkçe karakterler doğru mu? El yazısı parçaları okunabilir mi?
2. **S2 JSON:** Soru-cevap ayrımı doğru mu? Header eleyebildi mi? JSON şeması tutarlı mı?
3. **S3 el yazısı:** Cevaplar doğru sorulara eşleşiyor mu? NLP'ye besleyebileceğimiz kalitede mi?

**Karar matrisi:**
- ✅ S2 çalışıyorsa → production'da bunu kullan (en kolay yol)
- ✅ S1 + S3 çalışıyor ama S2 değil → S1 raw'ı al, Python tarafında regex ile parse et
- ❌ El yazısı hâlâ kötü → Qwen3-VL veya başka model dene